In [2]:
import sys

sys.path.insert(0, "..")  # repo root, where column_analysis.py lives
import numpy as np
from column_analysis import (
    load_data,
    column_profile,
    print_label_rates,
    find_exact_duplicates,
    find_correlated_pairs,
    find_determined_columns,
    choose_determined_drops,
    leakage_scan,
    save_decisions,
    summarize_decisions,
)

DATA_PATH = "../data/"  # x_train.csv, y_train.csv, x_test.csv
CACHE = "../data/cache.npz"
DECISIONS_PATH = "../decisions/column_decisions.csv"

x_train, x_test, y_train, train_ids, test_ids, names = load_data(DATA_PATH, CACHE)
y01 = (y_train == 1).astype(float)
profile = column_profile(x_train, x_test, y01, names)
P = {p["name"]: p for p in profile}
print(
    "train",
    x_train.shape,
    "| test",
    x_test.shape,
    "| positive rate",
    round(y01.mean(), 4),
)


train (328135, 321) | test (109379, 321) | positive rate 0.0883


In [3]:
miss = np.array([p["missing_train"] for p in profile])
print("columns with no missing value:", (miss == 0).sum())
for t in [0.01, 0.1, 0.5, 0.9]:
    print(f"columns with > {t:.0%} missing: {(miss > t).sum()}")
diff = max(abs(p["missing_train"] - p["missing_test"]) for p in profile)
print("largest train/test difference in missing rate:", round(diff, 4))
print("\nnearly complete columns (a no-NaN filter would drop them):")
for p in profile:
    if 0 < p["missing_train"] < 0.01:
        print(
            f"  {p['name']:10s} missing rows: {round(p['missing_train'] * len(y01))}, label gap {p['label_gap']:.3f}"
        )


columns with no missing value: 82
columns with > 1% missing: 221
columns with > 10% missing: 182
columns with > 50% missing: 147
columns with > 90% missing: 99
largest train/test difference in missing rate: 0.004

nearly complete columns (a no-NaN filter would drop them):
  GENHLTH    missing rows: 2, label gap 0.299
  PHYSHLTH   missing rows: 1, label gap 0.189
  MEDCOST    missing rows: 1, label gap 0.066
  CHECKUP1   missing rows: 1, label gap 0.063
  BPHIGH4    missing rows: 1, label gap 0.132
  CHCSCNCR   missing rows: 1, label gap 0.081
  HAVARTH3   missing rows: 1, label gap 0.101
  DIABETE3   missing rows: 5, label gap 0.181
  VETERAN3   missing rows: 1, label gap 0.101
  CHILDREN   missing rows: 3, label gap 0.085
  INCOME2    missing rows: 2471, label gap 0.115
  INTERNET   missing rows: 3207, label gap 0.104
  QSTLANG    missing rows: 14, label gap 0.014
  _STRWT     missing rows: 22, label gap nan
  _DRDXAR1   missing rows: 1883, label gap 0.101
  _LMTACT1   missing rows: 1

In [4]:
# Codebook sections "Record Identification" and "Weighting Variables" describe the interview, not the person.
METADATA = [
    "FMONTH",
    "IDATE",
    "IMONTH",
    "IDAY",
    "IYEAR",
    "DISPCODE",
    "SEQNO",
    "_PSU",
    "QSTVER",
    "QSTLANG",
    "_STSTR",
    "_STRWT",
    "_RAWRAKE",
    "_WT2RAKE",
    "_LLCPWT",
    "_CLLCPWT",
    "_DUALCOR",
    "_DUALUSE",
]
print("label gap of metadata columns (nan = too many distinct values to compute):")
for n in METADATA:
    print(
        f"  {n:10s} n_unique={P[n]['n_unique']:7d}  label gap={P[n]['label_gap']:.3f}"
    )
print(
    "\nfor comparison, median label gap over all columns:",
    round(np.nanmedian([p["label_gap"] for p in profile]), 3),
)

# QSTVER and _RAWRAKE have a gap close to the median. Check what that signal comes from.
i_raw, i_adult = names.index("_RAWRAKE"), names.index("NUMADULT")
both = ~np.isnan(x_train[:, i_raw]) & ~np.isnan(x_train[:, i_adult])
r = np.corrcoef(x_train[both, i_raw], x_train[both, i_adult])[0, 1]
print(f"\n_RAWRAKE vs NUMADULT (household size): r = {r:.3f}")
landline = ~np.isnan(x_train[:, names.index("CTELENUM")])
qst = x_train[:, names.index("QSTVER")]
print("QSTVER value, share of landline interviews, positive rate:")
for v in np.unique(qst):
    m = qst == v
    print(
        f"  {v:4g}  landline {landline[m].mean():.2f}  positive rate {y01[m].mean():.3f}"
    )


label gap of metadata columns (nan = too many distinct values to compute):
  FMONTH     n_unique=     12  label gap=0.010
  IDATE      n_unique=    414  label gap=nan
  IMONTH     n_unique=     12  label gap=0.008
  IDAY       n_unique=     31  label gap=0.029
  IYEAR      n_unique=      2  label gap=0.026
  DISPCODE   n_unique=      2  label gap=0.014
  SEQNO      n_unique=    182  label gap=nan
  _PSU       n_unique=    182  label gap=nan
  QSTVER     n_unique=      8  label gap=0.059
  QSTLANG    n_unique=      3  label gap=0.014
  _STSTR     n_unique=   1306  label gap=nan
  _STRWT     n_unique=   1304  label gap=nan
  _RAWRAKE   n_unique=     12  label gap=0.072
  _WT2RAKE   n_unique=   5528  label gap=nan
  _LLCPWT    n_unique= 216480  label gap=nan
  _CLLCPWT   n_unique=  20445  label gap=nan
  _DUALCOR   n_unique=    106  label gap=nan
  _DUALUSE   n_unique=      3  label gap=0.021

for comparison, median label gap over all columns: 0.072

_RAWRAKE vs NUMADULT (household size):

In [5]:
dups = find_exact_duplicates(x_train, names)
for a, b in dups:
    print(f"  {a:10s} is identical to {b}")
DUP_DROP = {a: f"exact duplicate of {b}" for a, b in dups}

# A column b is determined by a categorical column a if each answer to a always comes
# with the same value of b (for example _RFHLTH is a regrouping of GENHLTH).
# After one-hot encoding a, column b adds nothing, so it can be dropped.
determined = find_determined_columns(x_train, names, exclude=METADATA)
DETERMINED_DROP = {
    b: a
    for b, a in choose_determined_drops(determined, names).items()
    if b not in DUP_DROP
}
print(f"\n{len(DETERMINED_DROP)} columns are fully determined by another kept column:")
for b, a in DETERMINED_DROP.items():
    print(f"  {b:10s} determined by {a}")


  _PSU       is identical to SEQNO
  CELLFON2   is identical to CTELNUM1
  _CPRACE    is identical to _CRACE1

28 columns are fully determined by another kept column:
  _AIDTST3   determined by HIVTST6
  _RFSEAT3   determined by SEATBELT
  _RFSEAT2   determined by SEATBELT
  _PASTAE1   determined by _PAREC1
  _PA30021   determined by _PACAT1
  _PA300R2   determined by _PACAT1
  _PAINDX1   determined by _PA150R2
  _TOTINDA   determined by EXERANY2
  _VEG23     determined by _VEGETEX
  _FRT16     determined by _FRUITEX
  _VEGRESP   determined by _MISVEGN
  _FRTRESP   determined by _MISFRTN
  _RFSMOK3   determined by _SMOKER3
  _INCOMG    determined by INCOME2
  _EDUCAG    determined by EDUCA
  _RFBMI5    determined by _BMI5CAT
  _RACE_G1   determined by _RACE
  _RACEGR3   determined by _RACE
  _RACEG21   determined by _RACE
  _DRDXAR1   determined by HAVARTH3
  _CASTHM1   determined by _ASTHMS1
  _LTASTH1   determined by ASTHMA3
  _RFCHOL    determined by TOLDHI2
  _RFHYPE5   determined 

In [6]:
pairs = find_correlated_pairs(x_train, names, threshold=0.95)
for a, b, r, n in sorted(pairs, key=lambda t: -abs(t[2])):
    print(f"  {a:10s} ~ {b:10s} r={r:+.3f}  (rows with both: {n})")


  SEQNO      ~ _PSU       r=+1.000  (rows with both: 50000)
  HAVARTH3   ~ _DRDXAR1   r=+1.000  (rows with both: 49698)
  _CRACE1    ~ _CPRACE    r=+1.000  (rows with both: 7378)
  _STATE     ~ _STSTR     r=+1.000  (rows with both: 50000)
  IDATE      ~ IMONTH     r=+1.000  (rows with both: 50000)
  HTIN4      ~ HTM4       r=+1.000  (rows with both: 48092)
  SMOKDAY2   ~ _SMOKER3   r=+0.999  (rows with both: 20842)
  _PRACE1    ~ _MRACE1    r=+0.999  (rows with both: 50000)
  MAXVO2_    ~ FC60_      r=+0.998  (rows with both: 50000)
  EXERHMM2   ~ PADUR2_    r=+0.997  (rows with both: 21813)
  _VEGRESP   ~ _VEGETEX   r=-0.997  (rows with both: 50000)
  EXERHMM1   ~ PADUR1_    r=+0.996  (rows with both: 32703)
  _FRTRESP   ~ _FRUITEX   r=-0.996  (rows with both: 50000)
  EXERANY2   ~ _TOTINDA   r=+0.995  (rows with both: 46007)
  _DRNKWEK   ~ _RFDRHV5   r=+0.995  (rows with both: 50000)
  _RFSEAT2   ~ _RFSEAT3   r=+0.995  (rows with both: 50000)
  ASTHNOW    ~ _ASTHMS1   r=+0.994  (rows

In [7]:
for n, k, r in leakage_scan(x_train, y01, names):
    print(f"  {n}: answered by {k} people, {r:.1%} of them positive")
    print_label_rates(x_train, y01, names, n)


  HAREHAB1: answered by 773 people, 100.0% of them positive
HAREHAB1:  value   count   positive rate
          1      300   1.000
          2      458   1.000
          7        9   1.000
          9        6   1.000
        NaN   327362   0.086


In [8]:
nu = np.array([p["n_unique"] for p in profile])
print(
    "distinct values per column: 1:",
    (nu == 1).sum(),
    "| 2:",
    (nu == 2).sum(),
    "| 3-10:",
    ((nu >= 3) & (nu <= 10)).sum(),
    "| 11-20:",
    ((nu >= 11) & (nu <= 20)).sum(),
    "| >20:",
    (nu > 20).sum(),
)
print("\nborderline columns with 11-20 distinct values (check these by hand):")
for p in profile:
    if 11 <= p["n_unique"] <= 20:
        print(
            f"  {p['name']:10s} n={p['n_unique']:3d} top values: {[v for v, _ in p['top_values']]}"
        )


distinct values per column: 1: 6 | 2: 14 | 3-10: 199 | 11-20: 22 | >20: 80

borderline columns with 11-20 distinct values (check these by hand):
  FMONTH     n= 12 top values: [1.0, 10.0, 7.0, 4.0, 2.0, 3.0]
  IMONTH     n= 12 top values: [3.0, 11.0, 2.0, 1.0, 7.0, 12.0]
  NUMADULT   n= 16 top values: [2.0, 1.0, 3.0, 4.0, 5.0, 6.0]
  NUMMEN     n= 12 top values: [1.0, 0.0, 2.0, 3.0, 4.0, 5.0]
  NUMWOMEN   n= 11 top values: [1.0, 0.0, 2.0, 3.0, 4.0, 5.0]
  JOINPAIN   n= 13 top values: [5.0, 3.0, 4.0, 2.0, 8.0, 7.0]
  IMFVPLAC   n= 13 top values: [1.0, 5.0, 8.0, 3.0, 6.0, 9.0]
  WHRTST10   n= 11 top values: [1.0, 4.0, 8.0, 3.0, 2.0, 7.0]
  CRGVREL1   n= 17 top values: [1.0, 15.0, 6.0, 5.0, 2.0, 14.0]
  CRGVPRB1   n= 15 top values: [13.0, 5.0, 3.0, 8.0, 1.0, 7.0]
  ASDRVIST   n= 11 top values: [88.0, 1.0, 2.0, 3.0, 4.0, 98.0]
  ASRCHKUP   n= 18 top values: [88.0, 1.0, 2.0, 4.0, 3.0, 98.0]
  ADPLEASR   n= 17 top values: [88.0, 2.0, 14.0, 1.0, 3.0, 77.0]
  ADDOWN     n= 17 top values: [88.0

In [9]:
for n in ["GENHLTH", "PHYSHLTH", "_AGEG5YR", "INCOME2"]:
    print_label_rates(x_train, y01, names, n, min_count=200)
    print()
print("columns where a candidate code value covers >1% of rows")
print("(not always a code: _STATE 9 is a state, FMONTH 7 is July):")
for c in [7, 9, 77, 88, 99, 555, 777, 888, 999]:
    cols = [p["name"] for p in profile if p["codes"].get(c, 0) > 0.01]
    print(f"  code {c:>3}: {len(cols):3d} columns, e.g. {cols[:8]}")
print("\n_AGE80 is a computed age in years, so 77 there is a real age:")
print_label_rates(x_train, y01, names, "_AGE80", min_count=4000)


GENHLTH:  value   count   positive rate
          1    56937   0.021
          2   108089   0.042
          3   101880   0.091
          4    43349   0.193
          5    16992   0.319
          7      562   0.171
          9      324   0.142

PHYSHLTH:  value   count   positive rate
          1    13676   0.050
          2    18182   0.072
          3    10553   0.091
          4     5672   0.108
          5     9207   0.110
          6     1640   0.149
          7     5892   0.102
          8     1014   0.143
          9      258   0.132
         10     6754   0.153
         12      703   0.139
         14     3333   0.120
         15     6150   0.187
         20     4033   0.195
         21      830   0.139
         25     1568   0.206
         28      657   0.239
         29      296   0.280
         30    25233   0.229
         77     5587   0.163
         88   204264   0.059
         99     1520   0.159

_AGEG5YR:  value   count   positive rate
          1    18145   0.005
      

In [10]:
REDUNDANT = (
    {  # same concept in other units; not caught by cell 4 because they are continuous
        "_AGE65YR": "age group: keep _AGE80 (years) and _AGEG5YR (cell 5)",
        "_AGE_G": "age group: keep _AGE80 (years) and _AGEG5YR (cell 5)",
        "HEIGHT3": "raw height in mixed ft/in and m units: keep HTM4 (codebook)",
        "WEIGHT2": "raw weight in mixed lb/kg units: keep WTKG3 (codebook)",
        "HTIN4": "height in inches, same as HTM4 (r = 1.000, cell 5)",
    }
)
LEAKY = {
    "HAREHAB1": "label leakage: only asked after a heart attack, 100% positive (cell 6)"
}
SPECIAL = {  # column-specific codes found in cell 8 (computed columns)
    "_AGEG5YR": "14",
    "MAXVO2_": "999",
    "FC60_": "999",
    "DROCDY3_": "900",
    "_DRNKWEK": "99900",
}
RAW_CODE_OVERRIDE = {"DIABAGE2": "98 99"}  # the codebook differs from the generic rule
REAL_UNITS = {"HTM4", "HTIN4", "WTKG3"}
NOMINAL_OVERRIDE = {  # 11-20 values, but the numbers are labels, not amounts (cell 7)
    "CRGVREL1": "caregiver relationship",
    "CRGVPRB1": "type of care problem",
    "IMFVPLAC": "place of flu shot",
    "WHRTST10": "place of last HIV test",
    "_STATE": "US state code",
    "EXRACT11": "type of main exercise",
    "EXRACT21": "type of second exercise",
}


def is_computed(n):
    return n.startswith("_") or n.endswith("_") or n in REAL_UNITS


decisions = []
for p in profile:
    n, d = p["name"], {"name": p["name"], "missing_codes": "", "zero_codes": ""}
    if n in METADATA:
        if n in ("QSTVER", "_RAWRAKE"):
            reason = "survey design variable; its signal comes from phone type or household size, kept as person columns (cell 3)"
        else:
            reason = (
                "interview or survey weighting variable, not about the person (cell 3)"
            )
        d.update(action="drop", reason=reason)
    elif n in DUP_DROP:
        d.update(action="drop", reason=DUP_DROP[n] + " (cell 4)")
    elif n in DETERMINED_DROP:
        d.update(action="drop", reason=f"determined by {DETERMINED_DROP[n]} (cell 4)")
    elif n in REDUNDANT:
        d.update(action="drop", reason=REDUNDANT[n])
    elif n in LEAKY:
        d.update(action="drop", reason=LEAKY[n])
    elif p["n_unique"] <= 10:
        d.update(
            action="categorical",
            reason="<=10 distinct values: each answer, code and missing is its own category (cells 7-8)",
        )
    elif n in NOMINAL_OVERRIDE:
        d.update(
            action="categorical",
            reason=f"nominal labels ({NOMINAL_OVERRIDE[n]}), not amounts (cell 7)",
        )
    elif is_computed(n):
        d.update(
            action="continuous",
            missing_codes=SPECIAL.get(n, ""),
            reason="computed column in real units: only its own codes from SPECIAL (cell 8)",
        )
    else:
        mx = p["max"]
        if n in RAW_CODE_OVERRIDE:
            d.update(missing_codes=RAW_CODE_OVERRIDE[n], zero_codes="")
        elif mx <= 99:
            d.update(missing_codes="77 98 99" if mx >= 98 else "77 99", zero_codes="88")
        elif mx <= 999:
            d.update(missing_codes="777 999", zero_codes="555 888")
        if mx > 9999:
            d.update(
                action="continuous",
                missing_codes="777777 999999",
                zero_codes="",
                reason="REVIEW codebook: date coded as MMYYYY, convert to 'months ago' or drop",
            )
        elif p["min"] >= 101 and mx <= 999:
            d.update(
                action="continuous",
                reason="REVIEW codebook: unit-coded (1xx/2xx/3xx = per day/week/month), needs conversion",
            )
        else:
            d.update(
                action="continuous",
                reason="raw survey answer: 77/99 (777/999) -> missing, 88 (555/888) -> 0 (cell 8)",
            )
    if n in RAW_CODE_OVERRIDE:
        d[
            "reason"
        ] = "codebook p.21: 98 don't know, 99 refused; 77 is a real age (cell 8)"
    decisions.append(d)

save_decisions(decisions, DECISIONS_PATH)
summarize_decisions(decisions)
print(
    "\nColumns to review:",
    [d["name"] for d in decisions if d["reason"].startswith("REVIEW")],
)


categorical : 188 columns
       181  <=10 distinct values: each answer, code and missing is its own category (cells 7-8)
         1  nominal labels (US state code), not amounts (cell 7)
         1  nominal labels (type of main exercise), not amounts (cell 7)
         1  nominal labels (type of second exercise), not amounts (cell 7)
         1  nominal labels (place of flu shot), not amounts (cell 7)
         1  nominal labels (place of last HIV test), not amounts (cell 7)
         1  nominal labels (caregiver relationship), not amounts (cell 7)
         1  nominal labels (type of care problem), not amounts (cell 7)
continuous  : 79 columns
        32  computed column in real units: only its own codes from SPECIAL (cell 8)
        31  raw survey answer: 77/99 (777/999) -> missing, 88 (555/888) -> 0 (cell 8)
        13  REVIEW codebook: unit-coded (1xx/2xx/3xx = per day/week/month), needs conversion
         2  REVIEW codebook: date coded as MMYYYY, convert to 'months ago' or drop
     

In [11]:
from preprocessing import fit_preprocess, print_report
from column_analysis import load_decisions

params = fit_preprocess(x_train, names, load_decisions(DECISIONS_PATH))
print_report(params, show=("dropped", "continuous"))

SUMMARY: one-hot: 188, dropped: 54, continuous: 79 | final number of features: 1279

--- dropped (54) ---
  FMONTH                                                                  | interview or survey weighting variable, not about the person (cell 3)
  IDATE                                                                   | interview or survey weighting variable, not about the person (cell 3)
  IMONTH                                                                  | interview or survey weighting variable, not about the person (cell 3)
  IDAY                                                                    | interview or survey weighting variable, not about the person (cell 3)
  IYEAR                                                                   | interview or survey weighting variable, not about the person (cell 3)
  DISPCODE                                                                | interview or survey weighting variable, not about the person (cell 3)
  SEQNO           